# Dataset Preparation

**Prerequisites**
- `.env` at repo root with `XC_API_KEY=your_key`
- `uv sync` run in `building/`

In [1]:
%load_ext autoreload
%autoreload 2

# Parameters

In [2]:
N = 9
MAX_RECORDINGS = 10000

CLIPS_PER_SPECIES = 3500
MIN_XC_RECORDINGS = 100

ACTIVE_COLLECTIONS = ["diff_genus"] # ["diff_family", "diff_genus", "diff_species"]

# Select species

In [ ]:
from building.taxonomy import (
    get_species_with_recordings,
    select_same_genus,
    select_same_family,
    select_same_order,
)

pool = get_species_with_recordings(min_recordings=MIN_XC_RECORDINGS,dl_more=True)
print(f"Species pool: {len(pool)} species with {MIN_XC_RECORDINGS} XC recordings")

In [ ]:
# show what you can choose (based on MIN_XC_RECORDINGS and your N)
from building.taxonomy import get_potential_taxa
print("Potential genera (need >=N species):")
print(get_potential_taxa("genus", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential families (need >=N distinct genera):")
print(get_potential_taxa("family", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential orders (need >=N distinct families):")
print(get_potential_taxa("order", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))


Potential genera (need >=N species):
       genus  n_species  n_distinct_genus  n_distinct_family  n_distinct_order  min_top_n_rec
      Turdus         25                 1                  1                 1            352
Phylloscopus         22                 1                  1                 1            602
   Setophaga         22                 1                  1                 1            244
    Emberiza         14                 1                  1                 1            181
       Vireo         13                 1                  1                 1            176
Thamnophilus         12                 1                  1                 1            120
     Curruca         11                 1                  1                 1            198
      Anthus         11                 1                  1                 1            201
  Synallaxis         10                 1                  1                 1            114
Acrocephalus         10

In [ ]:
TARGET_GENUS  = "Turdus"
TARGET_FAMILY = "Fringillidae (Finches, Euphonias, and Allies)"
TARGET_ORDER  = "Passeriformes"

AVOID_DIFF_SPECIES = []
AVOID_DIFF_GENUS =["Loxia curvirostra","Acanthis flammea"]
AVOID_DIFF_FAMILY = []

In [ ]:
collections: dict[str, list[str]] = {}
diff_species = select_same_genus(TARGET_GENUS, N, pool, avoid=AVOID_DIFF_SPECIES)
diff_genus = select_same_family(TARGET_FAMILY, N, pool, avoid=AVOID_DIFF_GENUS)
diff_family = select_same_order(TARGET_ORDER, N, pool, avoid=AVOID_DIFF_FAMILY)
if "diff_species" in ACTIVE_COLLECTIONS:
    collections["diff_species"] = [s.scientific_name for s in diff_species]
if "diff_genus" in ACTIVE_COLLECTIONS:
    collections["diff_genus"] = [s.scientific_name for s in diff_genus]
if "diff_family" in ACTIVE_COLLECTIONS:
    collections["diff_family"] = [s.scientific_name for s in diff_family]

collections_to_use = {i_collection: collections[i_collection] for i_collection in ACTIVE_COLLECTIONS}

for coll, names in collections.items():
    print(f"\n{coll}:")
    for name in names:
        species = next(s for s in pool if s.scientific_name == name)
        print(f"  {name} ({species.num_recordings} recordings)")
print(f"\nACTIVE_COLLECTIONS (download + BirdNET): {ACTIVE_COLLECTIONS}")


diff_genus:
  Fringilla coelebs (4406 recordings)
  Chloris chloris (1588 recordings)
  Carduelis carduelis (1348 recordings)
  Pyrrhula pyrrhula (1041 recordings)
  Linaria cannabina (988 recordings)
  Serinus serinus (823 recordings)
  Spinus spinus (808 recordings)
  Carpodacus erythrinus (787 recordings)
  Loxia pytyopsittacus (688 recordings)

ACTIVE_COLLECTIONS (download + BirdNET): ['diff_genus']


# Download from Xeno-canto

In [ ]:
from building.download import download_and_process

for collection_name, species_names in collections.items():
    await download_and_process(
        species_names, 
        collection_name, 
        clips_per_species=CLIPS_PER_SPECIES, 
        auto_delete=True,)

Fetching available listings...
[Fringilla coelebs] 5000 clips, skipping.
[Chloris chloris] 5000 clips, skipping.
[Carduelis carduelis] 5000 clips, skipping.
[Pyrrhula pyrrhula] 3500 clips, skipping.
[Linaria cannabina] 3500 clips, skipping.
[Serinus serinus] 3500 clips, skipping.
[Spinus spinus] 3500 clips, skipping.
[Carpodacus erythrinus] 3500 clips, skipping.
[Loxia pytyopsittacus] 3500 clips, skipping.
Nothing to process.


# AudioSet ambient non_target

In [ ]:
from building.audioset import (
    AudioSetConfig,
    download_audioset_focus_classes,
    postprocess_audioset_clips,
    materialize_audioset_non_target,
)

AS_CLIPS_PER_CLASS = 400
AS_N_JOBS = 8

as_cfg = AudioSetConfig(clips_per_class=AS_CLIPS_PER_CLASS, n_jobs=AS_N_JOBS)
await download_audioset_focus_classes(as_cfg)
postprocess_audioset_clips(as_cfg)
for coll_name in collections_to_use:
    materialize_audioset_non_target(as_cfg, coll_name, total_clips=CLIPS_PER_SPECIES)


[audioset] downloading 24 classes: ['Aircraft', 'Bee, wasp, etc.', 'Chainsaw', 'Church bell', 'Cricket', 'Croak', 'Fly, housefly', 'Howl (wind)', 'Insect', 'Lawn mower', 'Light engine (high frequency)', 'Mosquito', 'Outside, rural or natural', 'Rain', 'Rain on surface', 'Raindrop', 'Rustling leaves', 'Stream', 'Thunder', 'Thunderstorm', 'Traffic noise, roadway noise', 'Truck', 'Wind', 'Wind noise (microphone)']


KeyError: 'Howl (wind)'

# BirdNET validation + dataset assembly

In [ ]:
from building.dataset import build_dataset

MAX_PER_CLASS = 2500
dataset_paths = {}
for coll_name, species_names in collections_to_use.items():
    dataset_paths[coll_name] = build_dataset(coll_name, species_names, clips_per_species=CLIPS_PER_SPECIES, max_class_size_training=MAX_PER_CLASS)
    print(f"  {coll_name}: {dataset_paths[coll_name]}")

Processing Fringilla_coelebs with 3500 clips
Training: 2450, Validation: 525, Testing: 525
Copied 2450 clips from Fringilla_coelebs to training
Copied 525 clips from Fringilla_coelebs to validation
Copied 525 clips from Fringilla_coelebs to testing
Processing Chloris_chloris with 3500 clips
Training: 2450, Validation: 525, Testing: 525
Copied 2450 clips from Chloris_chloris to training
Copied 525 clips from Chloris_chloris to validation
Copied 525 clips from Chloris_chloris to testing
Processing Carduelis_carduelis with 3500 clips
Training: 2450, Validation: 525, Testing: 525
Copied 2450 clips from Carduelis_carduelis to training
Copied 525 clips from Carduelis_carduelis to validation
Copied 525 clips from Carduelis_carduelis to testing
Processing Pyrrhula_pyrrhula with 3500 clips
Training: 2450, Validation: 525, Testing: 525
Copied 2450 clips from Pyrrhula_pyrrhula to training
Copied 525 clips from Pyrrhula_pyrrhula to validation
Copied 525 clips from Pyrrhula_pyrrhula to testing
Proc

# Sanity check

In [ ]:

for coll_name, root in dataset_paths.items():
    print(f"\n{coll_name}")
    for split in ("training", "validation", "testing"):
        split_dir = root / split
        if not split_dir.exists():
            continue
        species_counts = {
            d.name: len(list(d.glob("*.wav")))
            for d in sorted(split_dir.iterdir()) if d.is_dir()
        }
        total = sum(species_counts.values())
        print(f"  {split:12s} {total:5d} clips")
        for species, count in sorted(species_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"    {species:20s} {count:5d}")


diff_genus
  training     24500 clips
    Carduelis_carduelis   2450
    Carpodacus_erythrinus  2450
    Chloris_chloris       2450
    Fringilla_coelebs     2450
    Linaria_cannabina     2450
    Loxia_pytyopsittacus  2450
    Pyrrhula_pyrrhula     2450
    Serinus_serinus       2450
    Spinus_spinus         2450
    non_target            2450
  validation    5250 clips
    Carduelis_carduelis    525
    Carpodacus_erythrinus   525
    Chloris_chloris        525
    Fringilla_coelebs      525
    Linaria_cannabina      525
    Loxia_pytyopsittacus   525
    Pyrrhula_pyrrhula      525
    Serinus_serinus        525
    Spinus_spinus          525
    non_target             525
  testing       5250 clips
    Carduelis_carduelis    525
    Carpodacus_erythrinus   525
    Chloris_chloris        525
    Fringilla_coelebs      525
    Linaria_cannabina      525
    Loxia_pytyopsittacus   525
    Pyrrhula_pyrrhula      525
    Serinus_serinus        525
    Spinus_spinus          525
    n